# iRDT Training Notebook
Trains ECG and Audio classifiers on iRDT features with optimized delays.
Run cells in order. Cell 2 asks you to upload `irdt_core.py`.

## Cell 1: Install

In [ ]:
!pip install numba scikit-learn wfdb librosa scipy numpy importlib_resources tensorflow_datasets

## Cell 2: Upload irdt_core.py

In [ ]:
import os
if not os.path.exists('/content/irdt_core.py'):
    from google.colab import files
    files.upload()          # select irdt_core.py
print(os.path.exists('/content/irdt_core.py'))

## Cell 3: Imports and config

In [ ]:
import sys, time, pickle, io, contextlib
import numpy as np
from scipy.stats import skew, kurtosis
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier, VotingClassifier)
from sklearn.metrics import accuracy_score, classification_report

sys.path.append('/content')
from irdt_core import irdt_features, optimize_delays

SEED = 42
POPSIZE, MAXITER = 6, 20

# seg_len must give at least M+2 windows of size w, otherwise features are all zero
CFG = {
    'ECG':   dict(M=8,  w=64,  seg_len=640,   fixed=(1, 2, 4, 8, 16),     n_ch=6),
    'Audio': dict(M=16, w=128, seg_len=16000, fixed=(1, 2, 4, 8, 16, 32), n_ch=6),
}
for c in CFG.values():
    c['max_delay'] = c['w'] // 4          # keeps t±d inside the window

## Cell 4: Sanity check (features must be non-zero)

In [ ]:
for dom, c in CFG.items():
    x = np.random.uniform(-1, 1, c['seg_len']).astype(np.float32)
    f = irdt_features(x, M=c['M'], w=c['w'], chan=c['fixed'])
    print(dom, 'feature shape', f.shape, '| non-zero:', np.count_nonzero(f), '/', f.size)

## Cell 5: ECG loader (patient-independent split)

In [ ]:
ECG_RECORDS   = ['106', '119', '200', '203', '208', '210', '221', '233']
ECG_TEST_RECS = ['221', '233']

def load_ecg(per_rec=150):
    """Normal (0) vs PVC (1). Returns X, y, record id per beat."""
    import wfdb
    seg_len = CFG['ECG']['seg_len']; half = seg_len // 2
    rng = np.random.RandomState(SEED)
    X, y, g = [], [], []
    for rec in ECG_RECORDS:
        data = wfdb.rdrecord(rec, pn_dir='mitdb')
        ann  = wfdb.rdann(rec, 'atr', pn_dir='mitdb')
        sig = data.p_signal[:, 0].astype(np.float32)
        sig = sig - np.median(sig)
        sig = np.clip(sig / (np.percentile(np.abs(sig), 99.9) + 1e-8), -1, 1)  # into [-1,1]
        for label, symbol in ((0, 'N'), (1, 'V')):
            idx = [p for p, s in zip(ann.sample, ann.symbol)
                   if s == symbol and p - half >= 0 and p + half <= len(sig)]
            if len(idx) > per_rec:
                idx = rng.choice(idx, per_rec, replace=False)
            for p in idx:
                X.append(sig[p - half: p + half]); y.append(label); g.append(rec)
    return np.array(X), np.array(y), np.array(g)

## Cell 6: Audio loader

In [ ]:
def load_audio(split, n_per_class, seg_len=16000):
    """Full 1 s clips at 16 kHz. yes=0, no=1, up=2, down=3."""
    import tensorflow_datasets as tfds
    ds, info = tfds.load('speech_commands', split=split, with_info=True)
    target = ['yes', 'no', 'up', 'down']
    word2idx = {w: i for i, w in enumerate(target)}
    X, y = [], []
    counts = {i: 0 for i in range(len(target))}
    for ex in ds:
        word = info.features['label'].int2str(int(ex['label'].numpy()))
        if word not in word2idx: continue
        idx = word2idx[word]
        if counts[idx] >= n_per_class: continue
        a = ex['audio'].numpy().astype(np.float32)
        a = a - np.mean(a)
        a = a / (np.max(np.abs(a)) + 1e-8)
        a = np.pad(a, (0, seg_len - len(a))) if len(a) < seg_len else a[:seg_len]
        X.append(a); y.append(idx); counts[idx] += 1
        if all(v >= n_per_class for v in counts.values()): break
    return np.array(X), np.array(y)

## Cell 7: Feature functions and classifier

In [ ]:
def irdt_mat(X, domain, chan):
    c = CFG[domain]
    return np.array([irdt_features(x, M=c['M'], w=c['w'], chan=chan) for x in X],
                    dtype=np.float32)

def expand(F, M):
    """Raw iRDT + channel ratios per segment + mean/std over time (all derived from iRDT output)."""
    n = len(F)
    R = F.reshape(n, M, -1)
    ratio = R / (R.sum(axis=2, keepdims=True) + 1.0)
    return np.hstack([F, ratio.reshape(n, -1), R.mean(axis=1), R.std(axis=1)])

def irdt_stack(X, domain, chan_opt):
    """iRDT-only feature set (hardware-deployable)."""
    M = CFG[domain]['M']
    return np.hstack([expand(irdt_mat(X, domain, chan_opt), M),
                      expand(irdt_mat(X, domain, CFG[domain]['fixed']), M)])

def extra_features(X, domain):
    """Software-only handcrafted features."""
    out = []
    for x in X:
        if domain == 'Audio':
            import librosa
            mf = librosa.feature.mfcc(y=x, sr=16000, n_mfcc=20)
            d  = librosa.feature.delta(mf)
            f = np.concatenate([mf.mean(1), mf.std(1), d.std(1),
                                [np.mean(librosa.feature.zero_crossing_rate(x)),
                                 np.mean(librosa.feature.spectral_centroid(y=x, sr=16000))]])
        else:
            spec = np.abs(np.fft.rfft(x)) ** 2
            be = np.log1p([b.sum() for b in np.array_split(spec, 16)])
            mid = len(x) // 2
            q = x[mid - 40: mid + 40]
            f = np.concatenate([be, [np.std(x), skew(x), kurtosis(x),
                                     q.max(), q.min(), np.ptp(q),
                                     np.argmax(np.abs(q)), np.sum(np.abs(np.diff(q)))]])
        out.append(f)
    return np.nan_to_num(np.array(out, dtype=np.float32))

def make_ensemble():
    return VotingClassifier([
        ('et',  ExtraTreesClassifier(n_estimators=500, n_jobs=-1, random_state=SEED)),
        ('rf',  RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED)),
        ('hgb', HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED)),
    ], voting='soft')

## Cell 8: Load data (slow, run once)

In [ ]:
DATA = {}

Xe, ye, ge = load_ecg(per_rec=150)
te = np.isin(ge, ECG_TEST_RECS)
DATA['ECG'] = (Xe[~te], ye[~te], Xe[te], ye[te])

Xa_tr, ya_tr = load_audio('train', n_per_class=500)
Xa_te, ya_te = load_audio('test',  n_per_class=300)
DATA['Audio'] = (Xa_tr, ya_tr, Xa_te, ya_te)

for d, (a, b, c_, e) in DATA.items():
    print(d, '| train', len(a), np.bincount(b), '| test', len(c_), np.bincount(e))

## Cell 9: Optimize delays on train, evaluate on held-out test

In [ ]:
RESULTS = {}

def rf_acc(Ftr, ytr, Fte, yte):
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED).fit(Ftr, ytr)
    return accuracy_score(yte, rf.predict(Fte))

for domain, (Xtr, ytr, Xte, yte) in DATA.items():
    c = CFG[domain]
    print(f"\n{'='*64}\n  {domain}\n{'='*64}")

    # 1) baseline: fixed delays, raw iRDT, RF
    acc_fixed = rf_acc(irdt_mat(Xtr, domain, c['fixed']), ytr,
                       irdt_mat(Xte, domain, c['fixed']), yte)
    print(f"  [fixed delays {c['fixed']} + RF]          test acc = {acc_fixed:.4f}")

    # 2) delay optimisation on TRAIN only
    t0 = time.time()
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):          # hide the long DE log
        best_chan, proxy_acc, history = optimize_delays(
            list(Xtr), ytr, n_channels=c['n_ch'], M=c['M'], w=c['w'],
            max_delay=c['max_delay'], popsize=POPSIZE, maxiter=MAXITER, seed=SEED)
    print(f"  optimizer done in {time.time()-t0:.0f}s, best delays {best_chan}, train 1-NN {proxy_acc:.4f}")

    acc_opt = rf_acc(irdt_mat(Xtr, domain, best_chan), ytr,
                     irdt_mat(Xte, domain, best_chan), yte)
    print(f"  [optimized delays + RF]                  test acc = {acc_opt:.4f}")

    # 3) iRDT-only pipeline (deployable): expanded features + ensemble
    Itr, Ite = irdt_stack(Xtr, domain, best_chan), irdt_stack(Xte, domain, best_chan)
    m_irdt = make_ensemble().fit(Itr, ytr)
    pred = m_irdt.predict(Ite)
    acc_irdt = accuracy_score(yte, pred)
    print(f"  [iRDT-only expanded + ensemble]          test acc = {acc_irdt:.4f}")
    print(classification_report(yte, pred, digits=3))

    # 4) iRDT + handcrafted (software-only)
    Etr, Ete = extra_features(Xtr, domain), extra_features(Xte, domain)
    m_extra = make_ensemble().fit(np.hstack([Itr, Etr]), ytr)
    acc_extra = accuracy_score(yte, m_extra.predict(np.hstack([Ite, Ete])))
    print(f"  [iRDT + handcrafted + ensemble]          test acc = {acc_extra:.4f}")

    # 5) refit both on ALL data for deployment
    yall = np.concatenate([ytr, yte])
    Iall = np.vstack([Itr, Ite]); Eall = np.vstack([Etr, Ete])
    final_irdt  = make_ensemble().fit(Iall, yall)
    final_extra = make_ensemble().fit(np.hstack([Iall, Eall]), yall)

    RESULTS[domain] = {
        'fixed_chan': c['fixed'], 'best_chan': best_chan, 'M': c['M'], 'w': c['w'],
        'seg_len': c['seg_len'], 'history': history,
        'fixed_acc': acc_fixed, 'opt_acc': acc_opt,
        'irdt_acc': acc_irdt, 'extra_acc': acc_extra,
        'model': final_irdt,            # iRDT-only, hardware-deployable
        'model_extra': final_extra,     # needs extra_features(), software only
    }

## Cell 10: Summary

In [ ]:
print("\n" + "="*92)
print(f"{'Domain':<8}{'Fixed+RF':>10}{'Opt+RF':>10}{'iRDT ens.':>11}{'+Extras':>10}   Best delays")
print("="*92)
for d, r in RESULTS.items():
    print(f"{d:<8}{r['fixed_acc']:>10.4f}{r['opt_acc']:>10.4f}"
          f"{r['irdt_acc']:>11.4f}{r['extra_acc']:>10.4f}   {r['best_chan']}")
print("="*92)

## Cell 11: Save and download

In [ ]:
with open('/content/irdt_model.pkl', 'wb') as f:
    pickle.dump(RESULTS, f)
print("Saved /content/irdt_model.pkl")

from google.colab import files
files.download('/content/irdt_model.pkl')

## Cell 12: Convert ECG model to TFLite INT8 for Arduino
Self-contained — uploads irdt_model.pkl and irdt_core.py, rebuilds features, trains tiny Keras model, exports .tflite

In [ ]:
# ── Cell 12: TFLite conversion (self-contained, run independently) ──
!pip install -q wfdb numba
import os, sys, pickle
import numpy as np
import tensorflow as tf

# 1. Upload irdt_model.pkl and irdt_core.py if not already present
from google.colab import files
if not os.path.exists('/content/irdt_model.pkl'):
    print('Upload irdt_model.pkl')
    files.upload()
if not os.path.exists('/content/irdt_core.py'):
    print('Upload irdt_core.py')
    files.upload()

sys.path.append('/content')
from irdt_core import irdt_features

# 2. Load saved results
with open('/content/irdt_model.pkl', 'rb') as f:
    RESULTS = pickle.load(f)
print('Domains in model:', list(RESULTS.keys()))

# 3. Config (must match train_colab.ipynb Cell 3)
SEED = 42
CFG = {
    'ECG':   dict(M=8,  w=64,  seg_len=640,   fixed=(1, 2, 4, 8, 16),     n_ch=6),
    'Audio': dict(M=16, w=128, seg_len=16000, fixed=(1, 2, 4, 8, 16, 32), n_ch=6),
}

# 4. Rebuild iRDT features from ECG data
import wfdb
domain     = 'ECG'
res        = RESULTS[domain]
c          = CFG[domain]
best_chan   = res['best_chan']
fixed_chan  = c['fixed']
M, w       = c['M'], c['w']
seg_len    = c['seg_len']
half       = seg_len // 2

ECG_RECORDS = ['106', '119', '200', '203', '208', '210', '221', '233']
rng = np.random.RandomState(SEED)
X_all, y_all = [], []
for rec in ECG_RECORDS:
    data = wfdb.rdrecord(rec, pn_dir='mitdb')
    ann  = wfdb.rdann(rec, 'atr', pn_dir='mitdb')
    sig  = data.p_signal[:, 0].astype(np.float32)
    sig  = sig - np.median(sig)
    sig  = np.clip(sig / (np.percentile(np.abs(sig), 99.9) + 1e-8), -1, 1)
    for label, symbol in ((0, 'N'), (1, 'V')):
        idx = [p for p, s in zip(ann.sample, ann.symbol)
               if s == symbol and p - half >= 0 and p + half <= len(sig)]
        if len(idx) > 150: idx = rng.choice(idx, 150, replace=False)
        for p in idx:
            X_all.append(sig[p - half: p + half]); y_all.append(label)
X_all = np.array(X_all, dtype=np.float32)
y_all = np.array(y_all)
print(f'ECG samples: {len(X_all)}, classes: {np.bincount(y_all)}')

# 5. Build feature matrix (mirrors irdt_stack + expand from Cell 7)
def irdt_mat(X, chan):
    return np.array([irdt_features(x, M=M, w=w, chan=chan) for x in X], dtype=np.float32)

def expand(F):
    n = len(F); R = F.reshape(n, M, -1)
    ratio = R / (R.sum(axis=2, keepdims=True) + 1.0)
    return np.hstack([F, ratio.reshape(n, -1), R.mean(axis=1), R.std(axis=1)])

Iall = np.hstack([expand(irdt_mat(X_all, best_chan)),
                  expand(irdt_mat(X_all, fixed_chan))])
n_features = Iall.shape[1]
n_classes  = len(np.unique(y_all))
print(f'Feature size: {n_features}, Classes: {n_classes}')

# 6. Train tiny Keras model
tf.random.set_seed(SEED)
keras_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(n_features,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(n_classes, activation='softmax'),
])
keras_model.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])
history = keras_model.fit(Iall, y_all, epochs=50, batch_size=32,
                          validation_split=0.15, verbose=1)
print(f'Final train acc: {history.history["accuracy"][-1]:.4f}')

# 7. INT8 quantization
def representative_dataset():
    for i in range(min(200, len(Iall))):
        yield [Iall[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

with open('/content/irdt_ecg_model.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'TFLite model size: {len(tflite_model)/1024:.1f} KB')
print(f'N_FEATURES for Arduino sketch: {n_features}')

files.download('/content/irdt_ecg_model.tflite')